# ARC-AGI-3 HypothesisAgent — Kaggle Competition Notebook

**Architecture:** In-game learning via memory-augmented reasoning loop  
**Strategy:** Hypothesis → Observe → Confirm → BFS exploit  
**Budget:** 2 LLM calls/action max, adaptive reasoning effort, fast-path for explore/exploit phases  

Planning document version 3.0 — implements all 10 bug fixes + 4-part efficiency layer.

## STEP 1 — Install dependencies from wheels (no internet needed on Kaggle)

In [1]:
import os
from pathlib import Path

# Define the path to your offline wheels
offline_dir = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")

# Check if you are running offline in the Kaggle competition environment
if offline_dir.exists():
    print("Offline environment detected. Installing from local wheels...")
    os.system(
        'pip install --no-index '  # <--- Removed --no-deps here
        '--find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels '
        '--find-links /kaggle/input/notebooks/sonhujkjk/qwen-dependency/qwen_awq_wheels '
        '--find-links /kaggle/input/notebooks/sonhujkjk/qwen-dependency '
        'arc-agi python-dotenv autoawq qwen-vl-utils'
    )
else:
    print("Online/Dev environment detected. Installing from internet...")
    os.system('pip install arc-agi python-dotenv openai pillow numpy pydantic autoawq qwen-vl-utils -q')

print('Dependencies installed.')


Offline environment detected. Installing from local wheels...
Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels, /kaggle/input/notebooks/sonhujkjk/qwen-dependency/qwen_awq_wheels, /kaggle/input/notebooks/sonhujkjk/qwen-dependency
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/notebooks/sonhujkjk/qwen-dependency/qwen_awq_wheels/autoawq-0.2.9.tar.gz
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
Processing /kaggle/input/notebooks/sonhujkjk/qwen-dependency/qwen_awq_wheels/qwen_vl_utils-0.0.14-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
Processing

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.1

## STEP 2 — Copy ARC-AGI-3-Agents directory & wait for gateway

In [2]:
import os
import shutil

src = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
dst = '/kaggle/working/ARC-AGI-3-Agents'

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    if not os.path.exists(dst):
        print(f'Copying {src} → {dst}')
        shutil.copytree(src, dst)
    else:
        print(f'Already exists: {dst}')
    print('Waiting for gateway...')
    os.system(
        'curl --fail --retry 999 --retry-all-errors --retry-delay 5 '
        '--retry-max-time 600 http://gateway:8001/api/games'
    )
else:
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'Copying {src} → {dst}')
            shutil.copytree(src, dst)
        else:
            print(f'Already exists: {dst}')
    else:
        os.makedirs(dst, exist_ok=True)

# Always ensure our custom dirs exist (not in original repo)
os.makedirs(f'{dst}/agents/utils', exist_ok=True)
os.makedirs(f'{dst}/agents/templates', exist_ok=True)
print('Setup complete.')

Copying /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents → /kaggle/working/ARC-AGI-3-Agents
Setup complete.


## STEP 3 — Write `agents/utils/__init__.py`

In [3]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/__init__.py
"""ARC-AGI-3 utility modules for the HypothesisAgent."""
from .visual_renderer import grid_to_image, image_to_base64, make_image_block, image_diff, extract_json
from .frame_diff import FrameDelta, compute_delta, detect_player, get_full_hash, get_abstract_hash
from .rule_book import RuleBook, ActionEffect, ObjectDef
from .goal_detector import detect_goal_candidates
from .pathfinder import bfs_path, find_nearest_value
from .state_graph import StateGraph, StateNode
from .phase_controller import Phase, PhaseController
from .prompt_builder import PromptBuilder

__all__ = [
    'grid_to_image', 'image_to_base64', 'make_image_block', 'image_diff', 'extract_json',
    'FrameDelta', 'compute_delta', 'detect_player', 'get_full_hash', 'get_abstract_hash',
    'RuleBook', 'ActionEffect', 'ObjectDef',
    'detect_goal_candidates',
    'bfs_path', 'find_nearest_value',
    'StateGraph', 'StateNode',
    'Phase', 'PhaseController',
    'PromptBuilder',
]

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/__init__.py


## STEP 4 — Write `agents/utils/visual_renderer.py`

Extracted and consolidated from `multimodal.py`. Single source of truth for the palette, grid rendering, diffs, and JSON extraction.

In [4]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/visual_renderer.py
"""Visual utilities extracted from multimodal.py.
Single source of truth for palette, grid rendering, image diff, and JSON extraction.
Bug fix (Section 4.3 #5): use _SCALE=2 (128x128) instead of cell_size=40 (2560x2560).
"""
from __future__ import annotations

import base64
import io
import json
import re
from typing import Any, List, Sequence, Tuple

import numpy as np
from PIL import Image

# ── Canonical 16-colour palette (RGBA) ────────────────────────────────────────
# Extracted from multimodal.py — do NOT duplicate elsewhere.
_PALETTE: List[Tuple[int, int, int, int]] = [
    (0xFF, 0xFF, 0xFF, 0xFF),  # 0  White
    (0xCC, 0xCC, 0xCC, 0xFF),  # 1  Off-white
    (0x99, 0x99, 0x99, 0xFF),  # 2  Neutral light
    (0x66, 0x66, 0x66, 0xFF),  # 3  Neutral
    (0x33, 0x33, 0x33, 0xFF),  # 4  Off-black
    (0x00, 0x00, 0x00, 0xFF),  # 5  Black
    (0xE5, 0x3A, 0xA3, 0xFF),  # 6  Magenta
    (0xFF, 0x7B, 0xCC, 0xFF),  # 7  Magenta light
    (0xF9, 0x3C, 0x31, 0xFF),  # 8  Red
    (0x1E, 0x93, 0xFF, 0xFF),  # 9  Blue
    (0x88, 0xD8, 0xF1, 0xFF),  # 10 Blue light
    (0xFF, 0xDC, 0x00, 0xFF),  # 11 Yellow
    (0xFF, 0x85, 0x1B, 0xFF),  # 12 Orange
    (0x92, 0x12, 0x31, 0xFF),  # 13 Maroon
    (0x4F, 0xCC, 0x30, 0xFF),  # 14 Green
    (0xA3, 0x56, 0xD6, 0xFF),  # 15 Purple
]

_SCALE = 2          # 64 → 128 px  (was cell_size=40 → 2560 px, bug #5)
_TARGET_SIZE = 64 * _SCALE


def _validate_grid(grid: Sequence[Sequence[int]]) -> None:
    if len(grid) != 64 or any(len(row) != 64 for row in grid):
        raise ValueError('Grid must be 64×64.')
    if any(cell not in range(16) for row in grid for cell in row):
        raise ValueError('Grid values must be integers 0–15.')


def grid_to_image(grid: Sequence[Sequence[int]]) -> Image.Image:
    """Convert a 64×64 int grid to a 128×128 RGBA Pillow Image at _SCALE=2."""
    _validate_grid(grid)
    raw = bytearray()
    for row in grid:
        for idx in row:
            raw.extend(_PALETTE[idx])
    img = Image.frombytes('RGBA', (64, 64), bytes(raw))
    img = img.resize((_TARGET_SIZE, _TARGET_SIZE), Image.NEAREST)
    return img


def image_to_base64(img: Image.Image) -> str:
    """Return a base-64 encoded PNG string (no data-URL prefix)."""
    buffer = io.BytesIO()
    img.save(buffer, format='PNG', optimize=True)
    return base64.b64encode(buffer.getvalue()).decode('ascii')


def make_image_block(b64_string: str) -> dict[str, Any]:
    """Return the JSON block OpenAI expects for an inline base-64 image."""
    return {
        'type': 'image_url',
        'image_url': {'url': f'data:image/png;base64,{b64_string}'},
    }


def image_diff(
    img_a: Image.Image,
    img_b: Image.Image,
    highlight_rgb: Tuple[int, int, int] = (255, 0, 0),
) -> Image.Image:
    """Pixel-level diff: red pixels mark what changed between img_a and img_b."""
    a = np.asarray(img_a.convert('RGB'))
    b = np.asarray(img_b.convert('RGB'))
    if a.shape != b.shape:
        raise ValueError(f'Images must match; got {a.shape} vs {b.shape}')
    diff_mask = np.any(a != b, axis=-1)
    if not diff_mask.any():
        return Image.new('RGB', (a.shape[1], a.shape[0]), (0, 0, 0))
    diff_img = np.zeros_like(a)
    diff_img[diff_mask] = highlight_rgb
    return Image.fromarray(diff_img)


def extract_json(response_content: str) -> Any:
    """Extract JSON from a raw assistant string. Handles fences, inline, bare."""
    fence = re.search(r'```json\s*(\{.*?\})\s*```', response_content, re.S | re.I)
    if fence:
        return json.loads(fence.group(1))
    fence = re.search(r'```\s*(\{.*?\})\s*```', response_content, re.S)
    if fence:
        return json.loads(fence.group(1))
    start, end = response_content.find('{'), response_content.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('No JSON object detected in assistant reply')
    return json.loads(response_content[start:end + 1])

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/visual_renderer.py


## STEP 5 — Write `agents/utils/frame_diff.py`

Implements `FrameDelta`, `compute_delta`, player detection, and two-tier hashing (full + abstract/masked) per Sections 2.3 and 2.11.

In [5]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/frame_diff.py
"""Frame delta computation, player detection, and two-tier grid hashing.
Implements Sections 2.3 (FrameDelta) and 2.11 (masked-hash state abstraction).
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass, field
from typing import List, Optional, Set, Tuple

import numpy as np


@dataclass
class FrameDelta:
    """Structured diff between two consecutive frames (Section 2.3)."""
    cells_changed: int
    changed_cells: List[Tuple[int, int, int, int]]   # (row, col, old_val, new_val)
    player_delta: Optional[Tuple[int, int]]           # (dr, dc) or None
    level_changed: bool
    state_changed: bool
    new_values_seen: Set[int]                         # values that appeared first time
    grid_hash_before: str                             # exact SHA-1
    grid_hash_after: str                              # exact SHA-1
    abstract_hash_before: str                         # masked (decorative zeroed)
    abstract_hash_after: str                          # masked (decorative zeroed)


def get_full_hash(grid: List[List[int]]) -> str:
    """Exact SHA-1 hash for transition_cache (Section 2.10)."""
    return hashlib.sha1(str(grid).encode()).hexdigest()


def get_abstract_hash(grid: List[List[int]], decorative_values: Set[int]) -> str:
    """Hash with decorative values zeroed out — used for StateGraph (Section 2.11).
    Two frames that differ only in decorative elements collapse to the same node,
    preventing the 'every-frame-looks-new' problem in animated games.
    """
    masked = [[0 if v in decorative_values else v for v in row] for row in grid]
    return hashlib.sha1(str(masked).encode()).hexdigest()


def compute_delta(
    grid_before: List[List[int]],
    grid_after: List[List[int]],
    levels_before: int,
    levels_after: int,
    state_before: str,
    state_after: str,
    known_values: Set[int],
    decorative_values: Set[int],
    player_signature: Optional[List[List[int]]] = None,
) -> FrameDelta:
    """Compute structured FrameDelta between two 64×64 grids."""
    before = np.array(grid_before, dtype=np.int16)
    after = np.array(grid_after, dtype=np.int16)

    diff_mask = before != after
    changed_indices = np.argwhere(diff_mask)

    changed_cells: List[Tuple[int, int, int, int]] = [
        (int(r), int(c), int(before[r, c]), int(after[r, c]))
        for r, c in changed_indices
    ]

    # Detect new values
    after_values = set(int(v) for v in np.unique(after))
    new_values_seen = after_values - known_values

    # Detect player delta via coherent cluster movement
    player_delta = _detect_player_delta(changed_cells, before, after)

    full_before = get_full_hash(grid_before)
    full_after = get_full_hash(grid_after)
    abs_before = get_abstract_hash(grid_before, decorative_values)
    abs_after = get_abstract_hash(grid_after, decorative_values)

    return FrameDelta(
        cells_changed=int(diff_mask.sum()),
        changed_cells=changed_cells,
        player_delta=player_delta,
        level_changed=(levels_after != levels_before),
        state_changed=(state_before != state_after),
        new_values_seen=new_values_seen,
        grid_hash_before=full_before,
        grid_hash_after=full_after,
        abstract_hash_before=abs_before,
        abstract_hash_after=abs_after,
    )


def _detect_player_delta(
    changed_cells: List[Tuple[int, int, int, int]],
    before: np.ndarray,
    after: np.ndarray,
) -> Optional[Tuple[int, int]]:
    """Infer player movement from coherent cluster shift in changed cells.
    Returns (dr, dc) if a clear translation is detected, else None.
    Uses the heuristic that the player is a small connected cluster that
    disappears from old positions and reappears offset by (dr, dc).
    """
    if not changed_cells or len(changed_cells) > 50:
        return None

    # Find cells that 'left' (had a non-zero value, now floor/zero) vs 'arrived'
    left = [(r, c, ov) for r, c, ov, nv in changed_cells if ov != 0 and nv == 0]
    arrived = [(r, c, nv) for r, c, ov, nv in changed_cells if ov == 0 and nv != 0]

    if not left or not arrived or len(left) != len(arrived):
        return None

    # Check if all shifts are identical (rigid translation)
    dr_set = set()
    dc_set = set()
    for (r1, c1, _), (r2, c2, _) in zip(
        sorted(left), sorted(arrived)
    ):
        dr_set.add(r2 - r1)
        dc_set.add(c2 - c1)

    if len(dr_set) == 1 and len(dc_set) == 1:
        return (dr_set.pop(), dc_set.pop())
    return None


def detect_player(
    grid: List[List[int]],
    rule_book_signature: Optional[List[List[int]]] = None,
) -> Optional[Tuple[int, int]]:
    """Detect player position from grid. Returns (row, col) of player center or None.
    Looks for the player signature pattern if available, otherwise returns None.
    """
    if rule_book_signature is None:
        return None
    sig = np.array(rule_book_signature)
    h, w = sig.shape
    grid_np = np.array(grid)
    rows, cols = grid_np.shape
    for r in range(rows - h + 1):
        for c in range(cols - w + 1):
            if np.array_equal(grid_np[r:r+h, c:c+w], sig):
                return (r + h // 2, c + w // 2)
    return None


def classify_decorative_values(
    grid_history: List[List[List[int]]],
    player_deltas: List[Optional[Tuple[int, int]]],
    level_changed_flags: List[bool],
    state_changed_flags: List[bool],
    goal_candidate_values: Set[int],
) -> Set[int]:
    """Identify values that move on their own (enemies, particles, animations).
    A value is decorative if its cells change position across frames WITHOUT
    correlating with player movement, level change, or state change.
    Excludes current goal candidates (Section 2.11 bootstrapping rule).
    """
    if len(grid_history) < 3:
        return set()  # need at least 3 frames to classify reliably

    decorative: Set[int] = set()
    n = len(grid_history)

    all_values: Set[int] = set()
    for g in grid_history:
        for row in g:
            all_values.update(row)

    for val in all_values:
        if val == 0:
            continue  # floor/background
        if val in goal_candidate_values:
            continue  # never mask a goal candidate

        move_count = 0
        for i in range(1, n):
            if level_changed_flags[i] or state_changed_flags[i]:
                continue
            g_prev = np.array(grid_history[i - 1])
            g_curr = np.array(grid_history[i])
            prev_pos = set(map(tuple, np.argwhere(g_prev == val).tolist()))
            curr_pos = set(map(tuple, np.argwhere(g_curr == val).tolist()))
            if prev_pos != curr_pos and len(prev_pos) > 0:
                pd = player_deltas[i] if i < len(player_deltas) else None
                # If the value moved but the player didn't, or moved differently → decorative
                if pd is None:
                    move_count += 1
                else:
                    # Check if the value's movement matches player movement
                    offset_match = any(
                        (r + pd[0], c + pd[1]) in curr_pos for r, c in prev_pos
                    )
                    if not offset_match:
                        move_count += 1

        # If this value moved independently ≥ 2 times → decorative
        if move_count >= 2:
            decorative.add(val)

    return decorative

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/frame_diff.py


## STEP 6 — Write `agents/utils/rule_book.py`

Persistent in-session knowledge base (Section 2.4). Includes `transition_cache`, `decorative_values`, `reasoning_effort_log`, BFS path caching, and `to_compact_json()` for prompt injection.

In [6]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/rule_book.py
"""RuleBook: persistent in-session knowledge base (Section 2.4).
Formalises the nascent aggregated_findings field into a structured, persistable model.
Includes all efficiency-layer fields: transition_cache (2.10), decorative_values (2.11),
reasoning_effort_log (2.9), cached BFS path (2.7), locked_action (2.12).
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple


@dataclass
class ActionEffect:
    """What we know about a single action's effect."""
    action_name: str
    direction: Optional[str] = None        # 'up'|'down'|'left'|'right'|'interact'|None
    avg_cells_changed: float = 0.0
    effect_type: str = 'unknown'           # 'movement'|'state_change'|'no_effect'|'level_complete'
    requires_coordinates: bool = False     # True for ACTION6
    observation_count: int = 0


@dataclass
class ObjectDef:
    """A named object in the game world."""
    name: str
    values: List[int]          # grid cell values that make up this object
    behavior: str = ''         # natural language description
    locations: List[Tuple[int, int]] = field(default_factory=list)


@dataclass
class ProgressSnapshot:
    action_counter: int
    levels_completed: int
    confirmed_rules_count: int
    win_condition_known: bool


class RuleBook:
    """Persistent in-session knowledge accumulator (Section 2.4)."""

    def __init__(self) -> None:
        # ── Action semantics ──────────────────────────────────────────────────
        self.action_map: Dict[str, ActionEffect] = {}
        self.no_op_actions: List[str] = []

        # ── World model ───────────────────────────────────────────────────────
        self.impassable_values: List[int] = []
        self.passable_values: List[int] = []
        self.player_signature: Optional[List[List[int]]] = None
        self.player_position: Optional[Tuple[int, int]] = None

        # ── Object registry ───────────────────────────────────────────────────
        self.objects: Dict[str, ObjectDef] = {}

        # ── Goal model ────────────────────────────────────────────────────────
        self.win_condition: Optional[str] = None
        self.current_objective: str = 'Explore the environment and discover game rules.'
        self.goal_candidates: List[Tuple[int, float]] = []  # (value, score)

        # ── Knowledge quality ─────────────────────────────────────────────────
        self.confirmed_rules: List[str] = []
        self.tentative_rules: List[str] = []
        self.retracted_rules: List[str] = []

        # ── Level metadata ────────────────────────────────────────────────────
        self.levels_completed: int = 0
        self.action_budget_used: int = 0
        self.available_actions: List[str] = []
        self.known_values: Set[int] = set()

        # ── Efficiency layer ──────────────────────────────────────────────────
        # Section 2.10: (full_hash, action_name) → resulting full_hash
        self.transition_cache: Dict[Tuple[str, str], str] = {}
        # Section 2.11: values masked out for abstract hashing
        self.decorative_values: Set[int] = set()
        # Section 2.9: per-call effort used
        self.reasoning_effort_log: List[str] = []
        # Section 2.7: BFS path caching
        self.cached_bfs_path: Optional[List[str]] = None
        self.cached_bfs_goal: Optional[Tuple[int, int]] = None
        # Section 2.8: locked action for repeated execution without LLM
        self.locked_action: Optional[str] = None
        self.locked_action_repeats_remaining: int = 0
        self.locked_action_expected_outcome: Optional[str] = None
        # Rolling progress history for early-reset decisions
        self.progress_history: List[ProgressSnapshot] = []

    # ── Public helpers ────────────────────────────────────────────────────────

    def get_movement_action_for(self, direction: str) -> Optional[str]:
        """Return action name for a given direction string, e.g. 'up' → 'ACTION1'."""
        for name, effect in self.action_map.items():
            if effect.direction == direction:
                return name
        return None

    def confirmed_rules_as_bullets(self) -> str:
        if not self.confirmed_rules:
            return '- No confirmed rules yet.'
        return '\n'.join(f'- {r}' for r in self.confirmed_rules)

    def to_compact_json(self) -> str:
        """Compact representation for LLM prompt injection (< 500 tokens).
        Bug fix (Section 4.3 #6): inject full accumulated findings, not just last action.
        """
        return json.dumps({
            'win_condition': self.win_condition,
            'current_objective': self.current_objective,
            'confirmed_rules': self.confirmed_rules[-10:],    # keep recent
            'tentative_rules': self.tentative_rules[-5:],
            'action_map': {
                k: {'direction': v.direction, 'effect_type': v.effect_type}
                for k, v in self.action_map.items()
            },
            'no_op_actions': self.no_op_actions,
            'impassable_values': self.impassable_values,
            'player_position': list(self.player_position) if self.player_position else None,
            'goal_candidates': self.goal_candidates[:5],
            'decorative_values': list(self.decorative_values),
            'levels_completed': self.levels_completed,
            'action_budget_used': self.action_budget_used,
        }, indent=None, separators=(',', ':'))

    def update_from_analysis(self, updates: Dict[str, Any]) -> None:
        """Apply LLM-generated rule updates from the analysis call (Section 3, STEP 2)."""
        for rule in updates.get('confirmed_rules', []):
            if rule and rule not in self.confirmed_rules:
                self.confirmed_rules.append(rule)
                # Promote from tentative if present
                if rule in self.tentative_rules:
                    self.tentative_rules.remove(rule)

        for rule in updates.get('retracted_rules', []):
            if rule and rule not in self.retracted_rules:
                self.retracted_rules.append(rule)
                if rule in self.confirmed_rules:
                    self.confirmed_rules.remove(rule)
                if rule in self.tentative_rules:
                    self.tentative_rules.remove(rule)

        for rule in updates.get('new_tentative_rules', []):
            if rule and rule not in self.tentative_rules and rule not in self.confirmed_rules:
                self.tentative_rules.append(rule)

        if updates.get('win_condition'):
            self.win_condition = updates['win_condition']

        if updates.get('current_objective'):
            self.current_objective = updates['current_objective']

        # Update decorative values from LLM suggestion
        dv_update = updates.get('decorative_values_update')
        if dv_update and isinstance(dv_update, list):
            for v in dv_update:
                if isinstance(v, int):
                    # Only add if not a goal candidate
                    goal_vals = {gv for gv, _ in self.goal_candidates}
                    if v not in goal_vals:
                        self.decorative_values.add(v)

    def record_progress_snapshot(self, action_counter: int) -> None:
        self.progress_history.append(ProgressSnapshot(
            action_counter=action_counter,
            levels_completed=self.levels_completed,
            confirmed_rules_count=len(self.confirmed_rules),
            win_condition_known=(self.win_condition is not None),
        ))
        # Keep only last 20
        if len(self.progress_history) > 20:
            self.progress_history.pop(0)

    def carry_forward_for_new_level(self) -> None:
        """On level transition: keep movement/wall rules, reset goal model (Section 4, Phase 5)."""
        # Keep: impassable_values, passable_values, action_map, confirmed movement rules
        movement_rules = [r for r in self.confirmed_rules if any(
            kw in r.lower() for kw in ['move', 'wall', 'floor', 'pass', 'impass']
        )]
        # Reset goal-specific state
        self.win_condition = None
        self.current_objective = 'New level: re-identify the goal and win condition.'
        self.goal_candidates = []
        self.cached_bfs_path = None
        self.cached_bfs_goal = None
        self.locked_action = None
        self.locked_action_repeats_remaining = 0
        # Carry forward only movement rules
        self.confirmed_rules = movement_rules
        self.tentative_rules = []
        # transition_cache and decorative_values persist (Section 2.10 lifecycle)

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/rule_book.py


## STEP 7 — Write `agents/utils/goal_detector.py`

Multi-heuristic goal detection: rarity + edge proximity + change frequency (Section 2.6).

In [7]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/goal_detector.py
"""Extended goal detection: rarity + edge proximity + change frequency (Section 2.6).
Replaces single rare-color guess with a ranked list of goal candidates.
"""
from __future__ import annotations

from typing import List, Set, Tuple

import numpy as np


def detect_goal_candidates(
    grid_history: List[List[List[int]]],
    impassable_values: List[int],
    floor_values: Optional[List[int]] = None,
    weights: Tuple[float, float, float] = (0.5, 0.3, 0.2),
) -> List[Tuple[int, float]]:
    """Return [(value, goal_candidate_score), ...] sorted descending.
    Combines three signals:
      - rarity_weight    (0.5): rarer tiles are more likely to be goals
      - edge_weight      (0.3): goal tiles are often placed on grid edges
      - change_weight    (0.2): interactive objects (doors, keys) change frequently

    The weights (0.5/0.3/0.2) are empirical starting points; recalibrate in Phase 5
    using per-game goal_candidate_score vs confirmed win_condition data.
    """
    from typing import Optional  # local import to avoid top-level annotation issue

    if floor_values is None:
        floor_values = [0]

    if not grid_history:
        return []

    latest = grid_history[-1]
    rows, cols = len(latest), len(latest[0])
    grid_np = np.array(latest, dtype=np.int16)

    unique, counts = np.unique(grid_np, return_counts=True)
    excluded = set(impassable_values) | set(floor_values) | {0}

    w_rarity, w_edge, w_change = weights
    scores: dict[int, float] = {}

    for value, count in zip(unique.tolist(), counts.tolist()):
        if value in excluded:
            continue

        # 1. Rarity weight — rarer tiles score higher
        rarity_weight = 1.0 - min(count / 10.0, 1.0)

        # 2. Edge proximity weight — fraction of cells on outer 2-cell border
        positions = np.argwhere(grid_np == value)
        on_edge = sum(
            1 for r, c in positions
            if r < 2 or r >= rows - 2 or c < 2 or c >= cols - 2
        )
        edge_weight = on_edge / max(len(positions), 1)

        # 3. Change-frequency weight — interactive objects change often
        change_count = 0
        for i in range(1, len(grid_history)):
            prev = np.array(grid_history[i - 1], dtype=np.int16)
            curr = np.array(grid_history[i], dtype=np.int16)
            if prev.shape == curr.shape:
                changed = (prev != curr) & ((prev == value) | (curr == value))
                if changed.any():
                    change_count += 1
        change_weight = min(change_count / max(len(grid_history) - 1, 1), 1.0)

        scores[int(value)] = (
            w_rarity * rarity_weight
            + w_edge * edge_weight
            + w_change * change_weight
        )

    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def find_objects_by_value(
    grid: List[List[int]],
    value: int,
) -> List[Tuple[int, int]]:
    """Return all (row, col) positions of cells with the given value."""
    positions = []
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == value:
                positions.append((r, c))
    return positions


from typing import Optional  # needed at module level for the function signature

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/goal_detector.py


## STEP 8 — Write `agents/utils/pathfinder.py`

BFS pathfinder with path-caching discipline (Section 2.7). Zero LLM calls for pure movement.

In [8]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/pathfinder.py
"""BFS pathfinder with path-caching discipline (Section 2.7).
bfs_path() is always called with the latest impassable_values snapshot.
The result is cached in rule_book.cached_bfs_path — popped one step per turn.
"""
from __future__ import annotations

from collections import deque
from typing import List, Optional, Set, Tuple


# Action → (dr, dc) mapping — must match game's movement semantics.
# These are confirmed at Phase 1 exploration; BFS uses them after confirmation.
_MOVEMENT_ACTIONS = [
    (-1, 0, 'ACTION1'),   # Move Up
    (1, 0, 'ACTION2'),    # Move Down
    (0, -1, 'ACTION3'),   # Move Left
    (0, 1, 'ACTION4'),    # Move Right
]


def bfs_path(
    grid: List[List[int]],
    start: Tuple[int, int],
    target_value: int,
    impassable_values: List[int],
) -> Optional[List[str]]:
    """Find shortest path from start to nearest cell with target_value.
    Returns list of action names (e.g. ['ACTION4', 'ACTION1', 'ACTION1']), or None.
    Avoids cells whose value is in impassable_values.
    """
    rows, cols = len(grid), len(grid[0])
    start_r, start_c = start

    if grid[start_r][start_c] == target_value:
        return []  # already at goal

    impassable_set: Set[int] = set(impassable_values)
    queue: deque[Tuple[Tuple[int, int], List[str]]] = deque([(start, [])])
    visited: Set[Tuple[int, int]] = {start}

    while queue:
        (r, c), path = queue.popleft()

        for dr, dc, action in _MOVEMENT_ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                cell_val = grid[nr][nc]
                if cell_val == target_value:
                    return path + [action]  # reached goal
                if cell_val not in impassable_set:
                    visited.add((nr, nc))
                    queue.append(((nr, nc), path + [action]))

    return None  # no path found


def bfs_path_to_position(
    grid: List[List[int]],
    start: Tuple[int, int],
    target: Tuple[int, int],
    impassable_values: List[int],
) -> Optional[List[str]]:
    """Find shortest path from start to an exact target position."""
    rows, cols = len(grid), len(grid[0])
    if start == target:
        return []
    impassable_set: Set[int] = set(impassable_values)
    queue: deque[Tuple[Tuple[int, int], List[str]]] = deque([(start, [])])
    visited: Set[Tuple[int, int]] = {start}

    while queue:
        (r, c), path = queue.popleft()
        for dr, dc, action in _MOVEMENT_ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                if (nr, nc) == target:
                    return path + [action]
                if grid[nr][nc] not in impassable_set:
                    visited.add((nr, nc))
                    queue.append(((nr, nc), path + [action]))
    return None


def find_nearest_value(
    grid: List[List[int]],
    start: Tuple[int, int],
    target_value: int,
) -> Optional[Tuple[int, int]]:
    """BFS to find position of nearest cell with target_value (ignores impassable)."""
    rows, cols = len(grid), len(grid[0])
    queue: deque[Tuple[int, int]] = deque([start])
    visited: Set[Tuple[int, int]] = {start}
    while queue:
        r, c = queue.popleft()
        if grid[r][c] == target_value:
            return (r, c)
        for dr, dc, _ in _MOVEMENT_ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc))
    return None

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/pathfinder.py


## STEP 9 — Write `agents/utils/state_graph.py`

StateGraph keyed on `abstract_hash` (Section 2.5 + 2.11). Prevents revisiting equivalent states, enables BFS backtracking.

In [9]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/state_graph.py
"""StateGraph keyed on abstract_hash (Sections 2.5 + 2.11).
Prevents stuck-state cycles. Uses masked hashing so animated decorative elements
don't make every frame look like a new state.
"""
from __future__ import annotations

from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set


@dataclass
class StateNode:
    abstract_hash: str
    parent_hash: Optional[str] = None
    action_from_parent: Optional[str] = None
    tried_actions: Set[str] = field(default_factory=set)
    visit_count: int = 0


class StateGraph:
    """Directed graph of visited abstract game states and transitions tried."""

    def __init__(self) -> None:
        self.states: Dict[str, StateNode] = {}
        self._action_count: int = 0

    @property
    def size(self) -> int:
        return len(self.states)

    def add_state(
        self,
        abstract_hash: str,
        action_taken: Optional[str] = None,
        parent_hash: Optional[str] = None,
    ) -> StateNode:
        """Record a state visit. Creates node if new, updates visit count if existing."""
        if abstract_hash not in self.states:
            self.states[abstract_hash] = StateNode(
                abstract_hash=abstract_hash,
                parent_hash=parent_hash,
                action_from_parent=action_taken,
            )
        node = self.states[abstract_hash]
        node.visit_count += 1
        if action_taken and parent_hash:
            # Mark action as tried from the parent state
            if parent_hash in self.states:
                self.states[parent_hash].tried_actions.add(action_taken)
        return node

    def get_untried_actions(
        self,
        abstract_hash: str,
        available_actions: List[str],
    ) -> List[str]:
        """Return actions not yet tried from this state."""
        if abstract_hash not in self.states:
            return list(available_actions)
        tried = self.states[abstract_hash].tried_actions
        return [a for a in available_actions if a not in tried]

    def is_dead_end(self, abstract_hash: str, available_actions: List[str]) -> bool:
        """True if all available actions have been tried from this state."""
        untried = self.get_untried_actions(abstract_hash, available_actions)
        return len(untried) == 0

    def suggest_backtrack(
        self,
        current_hash: str,
        available_actions: List[str],
    ) -> List[str]:
        """BFS back through parent chain to find nearest state with untried actions.
        Returns the sequence of actions to get there (may be empty if stuck globally).
        """
        # BFS over parent pointers to find the nearest non-dead-end ancestor
        visited: Set[str] = {current_hash}
        queue: deque[List[str]] = deque([[current_hash]])

        while queue:
            path = queue.popleft()
            node_hash = path[-1]

            if node_hash != current_hash:
                if not self.is_dead_end(node_hash, available_actions):
                    # Found a reachable state with untried actions
                    # Return the reverse path as action names via parent chain
                    return self._path_to_actions(current_hash, node_hash)

            node = self.states.get(node_hash)
            if node and node.parent_hash and node.parent_hash not in visited:
                visited.add(node.parent_hash)
                queue.append(path + [node.parent_hash])

        return []  # globally stuck — accept partial score

    def _path_to_actions(self, from_hash: str, to_hash: str) -> List[str]:
        """Trace parent chain to get action sequence from from_hash to to_hash.
        Returns a list of action names to execute in sequence.
        """
        path: List[str] = []
        current = from_hash
        # Walk up the parent chain collecting actions
        visited: Set[str] = set()
        while current and current != to_hash and current not in visited:
            visited.add(current)
            node = self.states.get(current)
            if node and node.action_from_parent:
                path.append(node.action_from_parent)
                current = node.parent_hash or ''
            else:
                break
        # Reverse since we walked backwards
        return list(reversed(path))

    def get_visit_count(self, abstract_hash: str) -> int:
        node = self.states.get(abstract_hash)
        return node.visit_count if node else 0

    def is_stuck(
        self,
        recent_hashes: List[str],
        window: int = 5,
        threshold: int = 3,
    ) -> bool:
        """True if the same abstract state appears ≥ threshold times in the last window frames."""
        if len(recent_hashes) < window:
            return False
        from collections import Counter
        counts = Counter(recent_hashes[-window:])
        return counts.most_common(1)[0][1] >= threshold

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/state_graph.py


## STEP 10 — Write `agents/utils/phase_controller.py`

Phase determination + adaptive reasoning effort policy (Sections 2.8, 2.9, 3).

In [10]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/phase_controller.py
"""Phase controller and adaptive reasoning effort policy (Sections 2.8 + 2.9).
The effort policy is the PRIMARY defence against the ~30-hour budget risk:
  low/medium for early/validate turns, high only near wins or late in budget.
Expected to drop average effort latency 2-4x from flat 'high'.
"""
from __future__ import annotations

from enum import Enum
from typing import TYPE_CHECKING, Literal

if TYPE_CHECKING:
    from .rule_book import RuleBook
    from .state_graph import StateGraph


class Phase(str, Enum):
    EXPLORE = 'explore'      # Phase 1: systematic action probing
    VALIDATE = 'validate'    # Phase 2: rule formation, goal finding
    EXPLOIT = 'exploit'      # Phase 3: BFS goal execution
    RECOVERY = 'recovery'    # Phase 4: stuck-state recovery


class PhaseController:
    """Determines current game phase and per-call reasoning effort."""

    MAX_ACTIONS: int = 100  # Bug fix #3: was 400

    @staticmethod
    def determine_phase(
        rule_book: 'RuleBook',
        action_counter: int,
        state_graph: 'StateGraph',
        current_abstract_hash: str,
    ) -> Phase:
        """Select current phase based on exploration progress (Section 3)."""
        available = rule_book.available_actions
        n_available = len(available) if available else 4  # default 4 directional

        # Phase 1: probe all actions first
        if action_counter < n_available + 1:
            return Phase.EXPLORE

        # Still don't know how to win → validate
        if rule_book.win_condition is None:
            return Phase.VALIDATE

        # Not enough confirmed rules → validate
        if len(rule_book.confirmed_rules) < 3:
            return Phase.VALIDATE

        # Untried paths still exist → validate
        untried = state_graph.get_untried_actions(current_abstract_hash, available)
        if untried and action_counter < 75:
            return Phase.VALIDATE

        # Running out of budget → commit to exploit
        if action_counter > 75:
            return Phase.EXPLOIT

        return Phase.EXPLOIT if rule_book.win_condition else Phase.VALIDATE

    @staticmethod
    def determine_reasoning_effort(
        rule_book: 'RuleBook',
        phase: Phase,
        action_counter: int,
        state_graph: 'StateGraph',
        current_abstract_hash: str,
        max_actions: int = 100,
    ) -> Literal['low', 'medium', 'high']:
        """Select reasoning effort per-call to avoid the 30-hour budget overrun (Section 2.9).

        Table (Section 2.9):
          explore / exploit fast-path → no call (effort=n/a)
          early validate (counter < 20) → 'low'
          goal unknown OR stuck → 'medium'
          near confirmed win OR > 85% budget used → 'high'
          default → 'medium'
        """
        available = rule_book.available_actions or ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4']

        # Last-mile or confirmed win within reach → high
        if action_counter > 0.85 * max_actions:
            return 'high'
        if rule_book.win_condition and rule_book.cached_bfs_path is not None:
            return 'high'

        # Stuck state → medium (needs careful reasoning but not maximum)
        if state_graph.is_dead_end(current_abstract_hash, available):
            return 'medium'
        if rule_book.win_condition is None and action_counter > 20:
            return 'medium'

        # Early validate → low (cheap to revise)
        if action_counter < 20:
            return 'low'

        return 'medium'  # safe default

    @staticmethod
    def suggest_exploration_action(
        rule_book: 'RuleBook',
    ) -> str | None:
        """Return next untried action for systematic probing (Phase 1 fast path)."""
        tried = set(rule_book.action_map.keys())
        for action in rule_book.available_actions:
            if action not in tried:
                return action
        return None

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/phase_controller.py


## STEP 11 — Write `agents/utils/prompt_builder.py`

Phase-aware prompt templates for Analysis call and Action Selection call (Appendix A).

In [11]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/utils/prompt_builder.py
"""Phase-aware prompt templates for the two LLM calls (Appendix A).
Bug fix (Section 4.3 #7): removed hardcoded LockSmith hints from system prompt.
Bug fix (Section 4.3 #6): rule_book.to_compact_json() injected into every prompt.
"""
from __future__ import annotations

import json
from typing import TYPE_CHECKING, Any, Dict, List, Optional

if TYPE_CHECKING:
    from .frame_diff import FrameDelta
    from .rule_book import RuleBook
    from .phase_controller import Phase
    from .state_graph import StateGraph


ANALYSIS_SYSTEM = """You are a scientific game analyst. \
Analyse the frame delta and update the rule book. \
Be precise. Only add confirmed rules when you have clear evidence. \
Return ONLY valid JSON matching the schema specified."""

ACTION_SYSTEM = """You are a game-playing agent. \
Use confirmed rules to choose the best next action. \
Minimise total actions to win. \
Return exactly one tool call with the required fields."""


class PromptBuilder:
    """Builds LLM message lists for the two-call decision loop."""

    @staticmethod
    def build_analysis_messages(
        rule_book: 'RuleBook',
        delta: 'FrameDelta',
        last_action: Optional[str],
        expected_outcome: Optional[str],
        current_img_b64: str,
        diff_img_b64: Optional[str],
    ) -> List[Dict[str, Any]]:
        """Build messages for LLM Call 1 — Analysis (Section 3, STEP 2 + Appendix A.1)."""
        goal_candidates_str = json.dumps(rule_book.goal_candidates[:5])
        decorative_str = json.dumps(sorted(list(rule_book.decorative_values)))

        delta_summary = (
            f"Cells changed: {delta.cells_changed}\n"
            f"Player moved: {delta.player_delta}\n"
            f"Level changed: {delta.level_changed}\n"
            f"New values seen: {sorted(delta.new_values_seen)}"
        )

        user_text = f"""## Last Action Taken: {last_action or 'None'}
## Expected Outcome: {expected_outcome or 'None'}

## Frame Delta
{delta_summary}
Decorative values (masked for navigation, Section 2.11): {decorative_str}

## Goal Candidates (ranked by rarity+edge+change, Section 2.6)
{goal_candidates_str}

## Current Rule Book
{rule_book.to_compact_json()}

## Task
Return JSON only — no markdown, no preamble:
{{
  "delta_explanation": "what this delta tells us about game rules",
  "confirmed_rules": ["rule text", ...],
  "retracted_rules": ["rule text that was wrong", ...],
  "new_tentative_rules": ["hypothesis text", ...],
  "win_condition": "description or null",
  "current_objective": "what to do next",
  "decorative_values_update": [list of int values to classify as decorative, or null]
}}"""

        content: List[Dict[str, Any]] = [
            {
                'type': 'image_url',
                'image_url': {'url': f'data:image/png;base64,{current_img_b64}'},
            },
        ]
        if diff_img_b64:
            content.append({
                'type': 'image_url',
                'image_url': {'url': f'data:image/png;base64,{diff_img_b64}'},
            })
            content.append({'type': 'text', 'text': '[Above: current frame. Below that: red = changed pixels.]'})
        content.append({'type': 'text', 'text': user_text})

        return [
            {'role': 'system', 'content': ANALYSIS_SYSTEM},
            {'role': 'user', 'content': content},
        ]

    @staticmethod
    def build_action_messages(
        rule_book: 'RuleBook',
        phase: 'Phase',
        action_counter: int,
        max_actions: int,
        state_graph: 'StateGraph',
        current_abstract_hash: str,
        current_img_b64: str,
    ) -> List[Dict[str, Any]]:
        """Build messages for LLM Call 2 — Action Selection (Section 3, STEP 3 + Appendix A.2)."""
        available = rule_book.available_actions or ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'RESET']
        untried = state_graph.get_untried_actions(current_abstract_hash, available)

        user_text = f"""## Confirmed Rules
{rule_book.confirmed_rules_as_bullets()}

## Current Phase: {phase}
## Current Objective: {rule_book.current_objective}
## Actions Remaining: {max_actions - action_counter}
## Untried actions from current state: {untried}
## Player position: {rule_book.player_position}
## Win condition: {rule_book.win_condition or 'UNKNOWN — discover it'}

Choose the best next action. Return exactly one tool call:
  action: one of {available}
  x, y: grid coordinates 0-63 (ONLY if ACTION6, grid cell indices NOT pixel coords)
  reasoning: why this achieves the objective in ≤100 words
  expected_outcome: what should change in the next frame
  hypothesis_tested: which tentative rule this validates (or null)"""

        return [
            {'role': 'system', 'content': ACTION_SYSTEM},
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {'url': f'data:image/png;base64,{current_img_b64}'},
                    },
                    {'type': 'text', 'text': user_text},
                ],
            },
        ]

    @staticmethod
    def build_action_tools(
        available_actions: List[str],
    ) -> List[Dict[str, Any]]:
        """Build tool definitions for the action selection call.
        Bug fix (Section 4.3 #2): includes ACTION5, ACTION6, ACTION7.
        Bug fix (Section 4.3 #8): adds strict=True for consistency.
        """
        action_enum = available_actions + ['RESET']
        return [
            {
                'type': 'function',
                'function': {
                    'name': 'select_action',
                    'description': 'Choose the next game action to take.',
                    'strict': True,  # Bug fix #8
                    'parameters': {
                        'type': 'object',
                        'properties': {
                            'action': {
                                'type': 'string',
                                'enum': action_enum,
                                'description': 'The game action to take.',
                            },
                            'x': {
                                'type': 'integer',
                                'description': 'Grid column index 0-63 (only for ACTION6)',
                            },
                            'y': {
                                'type': 'integer',
                                'description': 'Grid row index 0-63 (only for ACTION6)',
                            },
                            'reasoning': {
                                'type': 'string',
                                'description': 'Why this action achieves the objective.',
                            },
                            'expected_outcome': {
                                'type': 'string',
                                'description': 'What should change in the next frame.',
                            },
                            'hypothesis_tested': {
                                'type': ['string', 'null'],
                                'description': 'Which tentative rule this validates, or null.',
                            },
                        },
                        'required': ['action', 'reasoning', 'expected_outcome', 'hypothesis_tested'],
                        'additionalProperties': False,
                    },
                },
            }
        ]

Writing /kaggle/working/ARC-AGI-3-Agents/agents/utils/prompt_builder.py


## STEP 12 — Write `agents/templates/hypothesis_agent.py`

The primary competition agent. Implements the full decision loop (Section 3) with all 10 bug fixes, the 4-part efficiency layer, and the 2-call-per-action LLM budget.

In [12]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/templates/hypothesis_agent.py
"""HypothesisAgent — primary ARC-AGI-3 competition agent.

Architecture: Section 4.1 (class hierarchy) and Section 3 (decision loop).
Bug fixes applied: all 10 from Section 4.3.
Efficiency layer: Sections 2.9 (adaptive effort), 2.10 (transition cache),
                  2.11 (masked hashing), 2.7 (BFS path caching).

Modified: Uses local Qwen2-VL 2B Instruct AWQ model instead of OpenAI API.
"""
from __future__ import annotations

import json
import logging
import os
import re
from typing import Any, Dict, List, Literal, Optional

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from arcengine import FrameData, GameAction, GameState

from ..agent import Agent
from ..utils.frame_diff import (
    FrameDelta, classify_decorative_values, compute_delta,
    detect_player, get_abstract_hash, get_full_hash,
)
from ..utils.goal_detector import detect_goal_candidates
from ..utils.pathfinder import bfs_path
from ..utils.phase_controller import Phase, PhaseController
from ..utils.prompt_builder import PromptBuilder
from ..utils.rule_book import RuleBook
from ..utils.state_graph import StateGraph
from ..utils.visual_renderer import (
    grid_to_image, image_diff, image_to_base64, make_image_block,
)

logger = logging.getLogger(__name__)

# Path to local Qwen2-VL 2B AWQ INT4 model
_LOCAL_MODEL_PATH = '/kaggle/input/models/marquis03/qwen2-vl/transformers/qwen2-vl-2b-instruct-awq/1'

# Singleton model/processor to avoid reloading across agent instances
_LOADED_MODEL = None
_LOADED_PROCESSOR = None


def _get_local_model():
    """Load the local AWQ model and processor (singleton pattern)."""
    global _LOADED_MODEL, _LOADED_PROCESSOR
    if _LOADED_MODEL is None:
        logger.info(f'Loading Qwen2-VL model from {_LOCAL_MODEL_PATH}...')
        _LOADED_PROCESSOR = AutoProcessor.from_pretrained(_LOCAL_MODEL_PATH)
        _LOADED_MODEL = Qwen2VLForConditionalGeneration.from_pretrained(
            _LOCAL_MODEL_PATH,
            torch_dtype=torch.float16,
            device_map='auto',
        )
        logger.info('Local model loaded successfully.')
    return _LOADED_MODEL, _LOADED_PROCESSOR


def _local_chat_completion(messages, max_new_tokens=1024):
    """Run chat completion using the local Qwen2-VL model.
    Decodes base64 images to PIL objects and uses Qwen2 processor natively.
    """
    import base64
    from io import BytesIO
    from PIL import Image
    model, processor = _get_local_model()
    
    text_messages = []
    images = []
    
    for msg in messages:
        role = msg.get('role', 'user')
        content = msg.get('content', '')
        if isinstance(content, list):
            new_content = []
            for part in content:
                if part.get('type') == 'image_url':
                    url = part['image_url']['url']
                    if 'base64,' in url:
                        b64_str = url.split('base64,')[1]
                    else:
                        b64_str = url
                    img = Image.open(BytesIO(base64.b64decode(b64_str))).convert('RGB')
                    images.append(img)
                    new_content.append({'type': 'image'})
                elif part.get('type') == 'text':
                    new_content.append({'type': 'text', 'text': part['text']})
            text_messages.append({'role': role, 'content': new_content})
        else:
            text_messages.append({'role': role, 'content': content})
    
    prompt = processor.apply_chat_template(
        text_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt],
        images=images if images else None,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return output_text


# Action name → direction string mapping (for RuleBook.action_map)
_ACTION_DIRECTIONS = {
    'ACTION1': 'up',
    'ACTION2': 'down',
    'ACTION3': 'left',
    'ACTION4': 'right',
    'ACTION5': 'interact',
    'ACTION6': 'click',
    'ACTION7': 'undo',
}

# All possible actions for build_tools (Bug fix #2: was missing ACTION5/6/7)
_ALL_ACTIONS = ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION5', 'ACTION6', 'ACTION7', 'RESET']


class HypothesisAgent(Agent):
    """Memory-augmented in-game learning agent for ARC-AGI-3.

    Decision loop (Section 3):
      Fast path  → return action without any LLM call (explore probe / BFS step / cache hit)
      STEP 1     → OBSERVE  (compute delta, update state graph, render visuals)
      STEP 2     → ANALYZE  (LLM call 1: update rule book)
      STEP 3     → SELECT   (LLM call 2: choose action via tool call)
      STEP 4     → BOOKKEEPING (cache, log, recorder)
    
    Uses local Qwen2-VL 2B Instruct AWQ model for inference.
    """

    # Bug fix #3: was 400 — now 100 to avoid score penalty and budget overrun
    MAX_ACTIONS = 100
    MODEL = 'qwen2-vl-2b-instruct-awq'  # local model identifier
    # Note: reasoning_effort is set dynamically per call (Section 2.9), not statically

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        # Pre-load local model (singleton — shared across instances)
        self._model, self._processor = _get_local_model()

        self.rule_book = RuleBook()
        self.state_graph = StateGraph()
        self.phase = Phase.EXPLORE
        self.phase_controller = PhaseController()

        # Per-turn state for comparison
        self._prev_grid: Optional[List[List[int]]] = None
        self._prev_img_b64: Optional[str] = None
        self._prev_full_hash: Optional[str] = None
        self._prev_abstract_hash: Optional[str] = None
        self._last_action_name: Optional[str] = None
        self._expected_outcome: Optional[str] = None
        self._prev_levels_completed: int = 0
        self._prev_state_name: str = ''

        # History for decorative value classification
        self._grid_history: List[List[List[int]]] = []
        self._player_delta_history: List[Optional[tuple]] = []
        self._level_changed_history: List[bool] = []
        self._state_changed_history: List[bool] = []
        self._abstract_hash_history: List[str] = []

    # ────────────────────────────────────────────────────────────────────────
    # Agent interface
    # ────────────────────────────────────────────────────────────────────────

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state == GameState.WIN

    def choose_action(
        self, frames: List[FrameData], latest_frame: FrameData
    ) -> GameAction:
        """Main decision loop. Returns a GameAction, takes ≤ 2 LLM calls."""

        # Handle NOT_PLAYED / GAME_OVER
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self._prev_grid = None
            return self._make_action('RESET', 'Game not started or over — resetting.')

        # Handle full_reset signal (level 0 restart)
        if latest_frame.full_reset:
            self._reset_for_new_game()
            return self._make_action('RESET', 'Full reset signal received.')

        # Sync rule_book metadata
        current_grid = latest_frame.frame[-1] if latest_frame.frame else []
        self._sync_available_actions(latest_frame)
        self.rule_book.levels_completed = latest_frame.levels_completed
        self.rule_book.action_budget_used = self.action_counter

        # Handle level transition: carry forward movement rules, reset goal model
        if latest_frame.levels_completed > self._prev_levels_completed:
            logger.info(f'Level up! {self._prev_levels_completed} → {latest_frame.levels_completed}')
            self.rule_book.carry_forward_for_new_level()
            self._prev_grid = None   # force delta reset
            self.state_graph = StateGraph()  # fresh graph per level
        self._prev_levels_completed = latest_frame.levels_completed

        # ── Compute hashes for current frame ─────────────────────────────────
        current_full_hash = get_full_hash(current_grid)
        current_abstract_hash = get_abstract_hash(current_grid, self.rule_book.decorative_values)

        # Update goal candidates
        self._grid_history.append(current_grid)
        if len(self._grid_history) > 20:
            self._grid_history.pop(0)
        self.rule_book.goal_candidates = detect_goal_candidates(
            self._grid_history,
            self.rule_book.impassable_values,
        )
        # Update known values
        for row in current_grid:
            self.rule_book.known_values.update(row)

        # ── TRANSITION CACHE lookup (Section 2.10) ────────────────────────────
        # If the next action in the BFS plan has a cached outcome, skip the LLM.
        if self.rule_book.cached_bfs_path and self._prev_full_hash:
            next_step = self.rule_book.cached_bfs_path[0]
            cache_key = (self._prev_full_hash, next_step)
            if cache_key in self.rule_book.transition_cache:
                # We know what will happen; pop it
                self.rule_book.cached_bfs_path.pop(0)
                self._bookkeeping(
                    current_full_hash, current_abstract_hash, next_step,
                    transition_cache_hit=True
                )
                action = self._make_action(next_step, f'Transition cache hit for {next_step}')
                return action

        # ── Update phase ──────────────────────────────────────────────────────
        self.phase = PhaseController.determine_phase(
            self.rule_book, self.action_counter, self.state_graph, current_abstract_hash
        )

        # ── FAST PATH — EXPLORE (Section 2.8 Phase 1) ────────────────────────
        if self.phase == Phase.EXPLORE:
            next_probe = PhaseController.suggest_exploration_action(self.rule_book)
            if next_probe:
                self._bookkeeping(current_full_hash, current_abstract_hash, next_probe)
                return self._make_action(next_probe, f'Systematic probe: testing {next_probe}')

        # ── FAST PATH — EXPLOIT via BFS (Section 2.8 Phase 3 + 2.7) ─────────
        if self.phase == Phase.EXPLOIT and self.rule_book.cached_bfs_path:
            next_step = self.rule_book.cached_bfs_path.pop(0)
            self._bookkeeping(current_full_hash, current_abstract_hash, next_step)
            return self._make_action(next_step, f'BFS path step: {next_step}')

        # If we're in exploit phase but have no cached path, compute BFS now
        if self.phase == Phase.EXPLOIT and self.rule_book.win_condition and self.rule_book.player_position:
            goal_val = self.rule_book.goal_candidates[0][0] if self.rule_book.goal_candidates else None
            if goal_val is not None:
                path = bfs_path(
                    current_grid,
                    self.rule_book.player_position,
                    goal_val,
                    self.rule_book.impassable_values,
                )
                if path:
                    self.rule_book.cached_bfs_path = path
                    self.rule_book.cached_bfs_goal = (0, 0)  # placeholder
                    next_step = self.rule_book.cached_bfs_path.pop(0)
                    self._bookkeeping(current_full_hash, current_abstract_hash, next_step)
                    return self._make_action(next_step, f'BFS computed + step: {next_step}')

        # ────────────────────────────────────────────────────────────────────
        # LLM PATH
        # ────────────────────────────────────────────────────────────────────

        # ── STEP 1: OBSERVE ───────────────────────────────────────────────────
        delta: Optional[FrameDelta] = None
        current_img = grid_to_image(current_grid)
        current_img_b64 = image_to_base64(current_img)
        diff_img_b64: Optional[str] = None

        if self._prev_grid is not None:
            prev_state_name = self._prev_state_name
            curr_state_name = latest_frame.state.name
            delta = compute_delta(
                self._prev_grid,
                current_grid,
                self._prev_levels_completed,
                latest_frame.levels_completed,
                prev_state_name,
                curr_state_name,
                self.rule_book.known_values,
                self.rule_book.decorative_values,
                self.rule_book.player_signature,
            )

            # Record histories for decorative classification
            self._player_delta_history.append(delta.player_delta)
            self._level_changed_history.append(delta.level_changed)
            self._state_changed_history.append(delta.state_changed)

            # Update player position
            if delta.player_delta is not None and self.rule_book.player_position:
                r, c = self.rule_book.player_position
                dr, dc = delta.player_delta
                self.rule_book.player_position = (r + dr, c + dc)

            # Update action_map for Phase 1 probing
            if self._last_action_name and self._last_action_name in self.rule_book.available_actions:
                self._update_action_map_from_delta(self._last_action_name, delta)

            # Compute diff image
            if self._prev_img_b64:
                from ..utils.visual_renderer import image_diff
                from PIL import Image
                import base64, io
                prev_bytes = base64.b64decode(self._prev_img_b64)
                prev_img = Image.open(io.BytesIO(prev_bytes))
                diff_img = image_diff(prev_img, current_img)
                diff_img_b64 = image_to_base64(diff_img)

            # Periodically update decorative values (every 5 frames after frame 10)
            if self.action_counter > 10 and self.action_counter % 5 == 0:
                goal_vals = {v for v, _ in self.rule_book.goal_candidates}
                new_decorative = classify_decorative_values(
                    self._grid_history[-15:],
                    self._player_delta_history[-15:],
                    self._level_changed_history[-15:],
                    self._state_changed_history[-15:],
                    goal_vals,
                )
                self.rule_book.decorative_values.update(new_decorative)

        # Update StateGraph with abstract hash
        self.state_graph.add_state(
            current_abstract_hash,
            action_taken=self._last_action_name,
            parent_hash=self._prev_abstract_hash,
        )
        self._abstract_hash_history.append(current_abstract_hash)

        # Detect stuck state
        if self.state_graph.is_stuck(self._abstract_hash_history):
            logger.warning(f'Stuck detected at action {self.action_counter}. Triggering recovery.')
            self.phase = Phase.RECOVERY
            self.rule_book.cached_bfs_path = None  # invalidate any stale BFS path

        # ── STEP 2: ANALYZE (LLM Call 1) ─────────────────────────────────────
        effort_1 = PhaseController.determine_reasoning_effort(
            self.rule_book, self.phase, self.action_counter,
            self.state_graph, current_abstract_hash, self.MAX_ACTIONS,
        )
        self.rule_book.reasoning_effort_log.append(effort_1)

        if delta is not None:
            analysis_messages = PromptBuilder.build_analysis_messages(
                self.rule_book, delta, self._last_action_name,
                self._expected_outcome, current_img_b64, diff_img_b64,
            )
            try:
                analysis_text = _local_chat_completion(analysis_messages, max_new_tokens=1000)
                from ..utils.visual_renderer import extract_json
                updates = extract_json(analysis_text)
                self.rule_book.update_from_analysis(updates)
                logger.debug(f'Analysis: {updates.get("delta_explanation", "")[:200]}')
            except Exception as e:
                logger.warning(f'Analysis LLM call failed: {e}')

        # ── STEP 3: SELECT ACTION (LLM Call 2) ───────────────────────────────
        effort_2 = PhaseController.determine_reasoning_effort(
            self.rule_book, self.phase, self.action_counter,
            self.state_graph, current_abstract_hash, self.MAX_ACTIONS,
        )
        self.rule_book.reasoning_effort_log.append(effort_2)

        available_action_names = [
            a.name if hasattr(a, 'name') else str(a)
            for a in (self.rule_book.available_actions or [])
        ] or _ALL_ACTIONS

        action_messages = PromptBuilder.build_action_messages(
            self.rule_book, self.phase, self.action_counter, self.MAX_ACTIONS,
            self.state_graph, current_abstract_hash, current_img_b64,
        )

        chosen_action_name = 'ACTION1'  # safe default
        action_x, action_y = 0, 0
        expected_outcome = 'Unknown'

        try:
            # Build a combined prompt that instructs the model to return JSON with action selection
            action_system_text = action_messages[0].get('content', '') if action_messages else ''
            action_user_content = action_messages[1].get('content', '') if len(action_messages) > 1 else ''
            if isinstance(action_user_content, list):
                # Extract text context to append instruction
                for i, part in enumerate(action_user_content):
                    if part.get('type') == 'text':
                        action_user_content[i]['text'] += (
                            '\n\nRespond with ONLY a JSON object (no markdown, no extra text):\n'
                            '{"action": "<ACTION_NAME>", "x": 0, "y": 0, '
                            '"reasoning": "...", "expected_outcome": "...", '
                            '"hypothesis_tested": null}'
                        )
                        break
            
            action_msgs_for_local = [
                {'role': 'system', 'content': action_system_text},
                {'role': 'user', 'content': action_user_content},
            ]
            action_text = _local_chat_completion(action_msgs_for_local, max_new_tokens=500)
            from ..utils.visual_renderer import extract_json
            args = extract_json(action_text)
            chosen_action_name = args.get('action', 'ACTION1')
            action_x = args.get('x', 0)
            action_y = args.get('y', 0)
            expected_outcome = args.get('expected_outcome', '')
            logger.info(
                f'[{self.phase}] Action {self.action_counter}: {chosen_action_name} | '
                f'effort={effort_2} | reasoning={args.get("reasoning", "")[:100]}'
            )
        except Exception as e:
            logger.warning(f'Action LLM call failed: {e}. Defaulting to recovery.')
            # Recovery: backtrack or try untried action
            untried = self.state_graph.get_untried_actions(current_abstract_hash, available_action_names)
            chosen_action_name = untried[0] if untried else 'RESET'

        # ── STEP 4: BOOKKEEPING ───────────────────────────────────────────────
        self._expected_outcome = expected_outcome
        self._bookkeeping(current_full_hash, current_abstract_hash, chosen_action_name)

        # Populate transition cache entry will be completed NEXT turn
        # (we know hash_before + action now; hash_after known next turn)
        self._pending_cache_key = (current_full_hash, chosen_action_name)

        # Update prev state
        self._prev_grid = current_grid
        self._prev_img_b64 = current_img_b64
        self._prev_full_hash = current_full_hash
        self._prev_abstract_hash = current_abstract_hash
        self._prev_state_name = latest_frame.state.name
        self._last_action_name = chosen_action_name

        # Record transition cache: previous turn's key → current hash
        if hasattr(self, '_pending_cache_key') and self._prev_full_hash:
            prev_key = getattr(self, '_prev_cache_key', None)
            if prev_key:
                self.rule_book.transition_cache[prev_key] = current_full_hash
        self._prev_cache_key = (current_full_hash, chosen_action_name)

        # Record debug log entry (Section 6.4)
        if hasattr(self, 'recorder'):
            log_entry = {
                'action_counter': self.action_counter,
                'phase': self.phase.value,
                'action': chosen_action_name,
                'reasoning_effort': effort_2,
                'abstract_hash': current_abstract_hash,
                'decorative_values': sorted(list(self.rule_book.decorative_values)),
                'goal_candidates': self.rule_book.goal_candidates[:3],
                'transition_cache_size': len(self.rule_book.transition_cache),
                'state_graph_size': self.state_graph.size,
                'confirmed_rules_count': len(self.rule_book.confirmed_rules),
                'win_condition': self.rule_book.win_condition,
                'expected_outcome': expected_outcome,
            }
            try:
                self.recorder.record(log_entry)
            except Exception:
                pass  # don't crash on recording failure

        # Build and return GameAction
        action = self._make_action(
            chosen_action_name,
            reasoning={
                'phase': self.phase.value,
                'reasoning_effort': effort_2,
                'expected_outcome': expected_outcome,
                'confirmed_rules_count': len(self.rule_book.confirmed_rules),
            },
            x=action_x,
            y=action_y,
        )
        return action

    # ────────────────────────────────────────────────────────────────────────
    # Private helpers
    # ────────────────────────────────────────────────────────────────────────

    def _make_action(
        self,
        action_name: str,
        reasoning: Any = '',
        x: int = 0,
        y: int = 0,
    ) -> GameAction:
        """Construct a GameAction from a name string.
        Bug fix #2: supports ACTION5, ACTION6, ACTION7 (were missing from ReasoningAgent).
        Bug fix for ACTION6 coordinate handling: grid indices, not pixel coords.
        """
        try:
            action = GameAction.from_name(action_name)
        except (ValueError, KeyError):
            logger.warning(f'Unknown action name: {action_name}, defaulting to ACTION1')
            action = GameAction.ACTION1

        if action_name == 'ACTION6' and action.is_complex():
            # Bug fix: ensure grid coordinates (0-63), NOT pixel coordinates
            grid_x = max(0, min(x, 63))
            grid_y = max(0, min(y, 63))
            action.set_data({'x': grid_x, 'y': grid_y})
        action.reasoning = reasoning
        return action

    def _sync_available_actions(self, latest_frame: FrameData) -> None:
        """Update rule_book.available_actions from frame."""
        if latest_frame.available_actions:
            names = []
            for a in latest_frame.available_actions:
                if hasattr(a, 'name'):
                    names.append(a.name)
                else:
                    names.append(str(a))
            self.rule_book.available_actions = names

    def _update_action_map_from_delta(self, action_name: str, delta: FrameDelta) -> None:
        """Update action_map entry based on observed delta (Phase 1 probing)."""
        from ..utils.rule_book import ActionEffect
        if action_name not in self.rule_book.action_map:
            self.rule_book.action_map[action_name] = ActionEffect(
                action_name=action_name,
                direction=_ACTION_DIRECTIONS.get(action_name),
            )
        effect = self.rule_book.action_map[action_name]
        effect.observation_count += 1

        # Determine effect type
        if delta.cells_changed == 0:
            effect.effect_type = 'no_effect'
            if action_name not in self.rule_book.no_op_actions:
                self.rule_book.no_op_actions.append(action_name)
        elif delta.level_changed:
            effect.effect_type = 'level_complete'
        elif delta.state_changed:
            effect.effect_type = 'state_change'
        else:
            effect.effect_type = 'movement'

        # Running average of cells changed
        n = effect.observation_count
        effect.avg_cells_changed = (
            (effect.avg_cells_changed * (n - 1) + delta.cells_changed) / n
        )

        # Infer impassable values: if movement attempted but player didn't move
        if (
            effect.effect_type == 'movement'
            and delta.player_delta == (0, 0)
            and self._prev_grid is not None
        ):
            # The cell the player tried to enter was impassable
            dr, dc = {
                'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1),
            }.get(effect.direction or '', (0, 0))
            if self.rule_book.player_position and (dr, dc) != (0, 0):
                r, c = self.rule_book.player_position
                target_r, target_c = r + dr, c + dc
                if 0 <= target_r < 64 and 0 <= target_c < 64:
                    target_val = self._prev_grid[target_r][target_c]
                    if target_val not in self.rule_book.impassable_values:
                        self.rule_book.impassable_values.append(target_val)
                        logger.info(f'Discovered impassable value: {target_val}')

    def _bookkeeping(
        self,
        current_full_hash: str,
        current_abstract_hash: str,
        action_name: str,
        transition_cache_hit: bool = False,
    ) -> None:
        """STEP 4 bookkeeping for fast-path actions (Section 3, STEP 4)."""
        self.state_graph.add_state(
            current_abstract_hash,
            action_taken=action_name,
            parent_hash=self._prev_abstract_hash,
        )
        self.rule_book.record_progress_snapshot(self.action_counter)
        self._last_action_name = action_name
        self._prev_full_hash = current_full_hash
        self._prev_abstract_hash = current_abstract_hash

    def _reset_for_new_game(self) -> None:
        """Reset all per-game state for a full restart."""
        self.rule_book = RuleBook()
        self.state_graph = StateGraph()
        self.phase = Phase.EXPLORE
        self._prev_grid = None
        self._prev_img_b64 = None
        self._prev_full_hash = None
        self._prev_abstract_hash = None
        self._last_action_name = None
        self._expected_outcome = None
        self._prev_levels_completed = 0
        self._grid_history = []
        self._player_delta_history = []
        self._level_changed_history = []
        self._state_changed_history = []
        self._abstract_hash_history = []


Writing /kaggle/working/ARC-AGI-3-Agents/agents/templates/hypothesis_agent.py


## STEP 13 — Apply Bug Fixes to `reasoning_agent.py`

All 10 bugs from Section 4.3. The patched file is written over the competition copy.

In [13]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/templates/reasoning_agent_fixed.py
"""reasoning_agent.py with all 10 bug fixes from Section 4.3 applied.
Key fixes:
  #1  latest_frame.score → latest_frame.levels_completed
  #2  Added ACTION5, ACTION6, ACTION7 to build_functions()
  #3  MAX_ACTIONS 400 → 100
  #4  reasoning_effort now passed to API (dynamically via PhaseController)
  #5  cell_size=40 (2560px) → cell_size=8 (512px)
  #6  aggregated_findings re-injected via rule_book.to_compact_json()
  #7  Removed hardcoded LockSmith hints from system prompt
  #8  Added 'strict': True to build_tools()
  #9  Model loaded once in __init__ (not re-created per action)
  #10 (In llm_agents.py) FIFO drop fix — implemented in push_message below

Modified: Uses local Qwen2-VL 2B Instruct AWQ model instead of OpenAI API.
"""
import base64
import io
import json
import logging
import re
import textwrap
from typing import Any, Dict, List, Literal

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from arcengine import FrameData, GameAction
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel, Field

from .llm_agents import ReasoningLLM

logger = logging.getLogger(__name__)

# Path to local Qwen2-VL 2B AWQ INT4 model
_LOCAL_MODEL_PATH = '/kaggle/input/models/marquis03/qwen2-vl/transformers/qwen2-vl-2b-instruct-awq/1'

# Singleton model/processor to avoid reloading
_RA_LOADED_MODEL = None
_RA_LOADED_PROCESSOR = None


def _get_local_model_ra():
    """Load the local AWQ model and processor (singleton pattern)."""
    global _RA_LOADED_MODEL, _RA_LOADED_PROCESSOR
    if _RA_LOADED_MODEL is None:
        logger.info(f'Loading local model from {_LOCAL_MODEL_PATH}...')
        _RA_LOADED_PROCESSOR = AutoProcessor.from_pretrained(_LOCAL_MODEL_PATH)
        _RA_LOADED_MODEL = Qwen2VLForConditionalGeneration.from_pretrained(
            _LOCAL_MODEL_PATH,
            torch_dtype=torch.float16,
            device_map='auto',
        )
        logger.info('Local model loaded successfully.')
    return _RA_LOADED_MODEL, _RA_LOADED_PROCESSOR


def _local_chat_completion_ra(messages, max_new_tokens=1024):
    """Run chat completion using the local Qwen2-VL model.
    Decodes base64 images to PIL objects and uses Qwen2 processor natively.
    """
    model, processor = _get_local_model_ra()
    
    text_messages = []
    images = []
    
    for msg in messages:
        role = msg.get('role', 'user')
        content = msg.get('content', '')
        if isinstance(content, list):
            new_content = []
            for part in content:
                if part.get('type') == 'image_url':
                    url = part['image_url']['url']
                    if 'base64,' in url:
                        b64_str = url.split('base64,')[1]
                    else:
                        b64_str = url
                    img = Image.open(io.BytesIO(base64.b64decode(b64_str))).convert('RGB')
                    images.append(img)
                    new_content.append({'type': 'image'})
                elif part.get('type') == 'text':
                    new_content.append({'type': 'text', 'text': part['text']})
            text_messages.append({'role': role, 'content': new_content})
        else:
            text_messages.append({'role': role, 'content': content})
    
    prompt = processor.apply_chat_template(
        text_messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[prompt],
        images=images if images else None,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return output_text


def _extract_json_from_text(text):
    """Extract JSON object from generated text."""
    # Try fenced JSON block first
    fence = re.search(r'```json\\s*(\\{.*?\\})\\s*```', text, re.S | re.I)
    if fence:
        return json.loads(fence.group(1))
    fence = re.search(r'```\\s*(\\{.*?\\})\\s*```', text, re.S)
    if fence:
        return json.loads(fence.group(1))
    start, end = text.find('{'), text.rfind('}')
    if start != -1 and end != -1 and end > start:
        return json.loads(text[start:end + 1])
    raise ValueError('No JSON object detected in model reply')


class ReasoningActionResponse(BaseModel):
    name: Literal[
        'ACTION1', 'ACTION2', 'ACTION3', 'ACTION4',
        'ACTION5', 'ACTION6', 'ACTION7', 'RESET'  # Bug fix #2: all actions
    ] = Field(description='The action to take.')
    reason: str = Field(description='Detailed reasoning', min_length=10, max_length=2000)
    short_description: str = Field(description='Brief description', min_length=5, max_length=500)
    hypothesis: str = Field(description='Current hypothesis about game mechanics', min_length=10, max_length=2000)
    aggregated_findings: str = Field(description='Summary of all discoveries so far', min_length=10, max_length=2000)


class ReasoningAgentFixed(ReasoningLLM):
    """Bug-fixed ReasoningAgent — all 10 fixes from Section 4.3 applied.
    Uses local Qwen2-VL 2B Instruct AWQ model."""

    MAX_ACTIONS = 100           # Bug fix #3: was 400
    DO_OBSERVATION = True
    MODEL = 'qwen2-vl-2b-instruct-awq'  # local model identifier
    MESSAGE_LIMIT = 5
    REASONING_EFFORT = 'medium'  # Bug fix #4: default; PhaseController sets per-call
    ZONE_SIZE = 16

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.history: List[ReasoningActionResponse] = []
        self.screen_history: List[bytes] = []
        self.max_screen_history = 10
        # Load local model (singleton — shared across instances)
        self._model, self._processor = _get_local_model_ra()

    def clear_history(self) -> None:
        self.history = []
        self.screen_history = []

    def generate_grid_image_with_zone(self, grid: List[List[int]], cell_size: int = 8) -> bytes:
        # Bug fix #5: default cell_size=8 gives 512×512 (was 40 → 2560×2560)
        if not grid or not grid[0]:
            img = Image.new('RGB', (200, 200), color='black')
            buffer = io.BytesIO()
            img.save(buffer, format='PNG')
            return buffer.getvalue()

        height = len(grid)
        width = len(grid[0])
        img = Image.new('RGB', (width * cell_size, height * cell_size), color='white')
        draw = ImageDraw.Draw(img)

        key_colors = {
            0: '#FFFFFF', 1: '#CCCCCC', 2: '#999999', 3: '#666666',
            4: '#333333', 5: '#000000', 6: '#E53AA3', 7: '#FF7BCC',
            8: '#F93C31', 9: '#1E93FF', 10: '#88D8F1', 11: '#FFDC00',
            12: '#FF851B', 13: '#921231', 14: '#4FCC30', 15: '#A356D6',
        }

        for y in range(height):
            for x in range(width):
                color = key_colors.get(grid[y][x], '#888888')
                draw.rectangle(
                    [x * cell_size, y * cell_size, (x + 1) * cell_size, (y + 1) * cell_size],
                    fill=color, outline='#000000', width=1,
                )

        buffer = io.BytesIO()
        img.save(buffer, format='PNG')
        return buffer.getvalue()

    def build_functions(self) -> list[dict[str, Any]]:
        schema = ReasoningActionResponse.model_json_schema()
        schema['properties'].pop('name', None)
        if 'required' in schema:
            schema['required'] = [r for r in schema['required'] if r != 'name']

        # Bug fix #2: all 7 actions + RESET (was missing ACTION5, ACTION6, ACTION7)
        all_actions = [
            GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3, GameAction.ACTION4,
            GameAction.ACTION5, GameAction.ACTION6, GameAction.ACTION7, GameAction.RESET,
        ]
        return [
            {'name': a.name, 'description': f'Take action {a.name}', 'parameters': schema}
            for a in all_actions
        ]

    def build_tools(self) -> list[dict[str, Any]]:
        tools = []
        for f in self.build_functions():
            tools.append({
                'type': 'function',
                'function': {
                    'name': f['name'],
                    'description': f['description'],
                    'parameters': f.get('parameters', {}),
                    'strict': True,  # Bug fix #8: was missing
                },
            })
        return tools

    def build_user_prompt(self, latest_frame: FrameData) -> str:
        # Bug fix #7: removed hardcoded LockSmith hints
        return textwrap.dedent("""
            You are playing an unknown 2D puzzle game. Your goal is to discover the rules
            and win in as few actions as possible.

            Available actions:
            - RESET: start/restart the game
            - ACTION1: Move Up (or unknown effect — probe to discover)
            - ACTION2: Move Down
            - ACTION3: Move Left
            - ACTION4: Move Right
            - ACTION5: Interact / Spacebar
            - ACTION6: Click on grid cell (provide x, y as grid indices 0-63)
            - ACTION7: Undo

            Each turn you see the previous screen and current screen.
            Compare them carefully. Identify what changed and why.

            Build a hypothesis. Test it. Store confirmed findings.
            Your aggregated_findings must summarise ALL discoveries so far
            so your colleagues can understand the game from that alone.
        """).strip()

    def call_llm_with_structured_output(self, messages: List[Dict[str, Any]]) -> ReasoningActionResponse:
        # Build available action names for the JSON instruction
        available_actions = ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4',
                             'ACTION5', 'ACTION6', 'ACTION7', 'RESET']
        # Append JSON format instruction to the last user message
        augmented_messages = list(messages)
        json_instruction = (
            '\n\nYou MUST respond with ONLY a valid JSON object (no markdown, no extra text).\n'
            'Use this exact schema:\n'
            '{\n'
            '  "name": "<one of: ' + ', '.join(available_actions) + '>",\n'
            '  "reason": "detailed reasoning (10-2000 chars)",\n'
            '  "short_description": "brief description (5-500 chars)",\n'
            '  "hypothesis": "current hypothesis about game mechanics (10-2000 chars)",\n'
            '  "aggregated_findings": "summary of all discoveries so far (10-2000 chars)"\n'
            '}'
        )
        if augmented_messages and isinstance(augmented_messages[-1].get('content'), str):
            augmented_messages[-1] = dict(augmented_messages[-1])
            augmented_messages[-1]['content'] += json_instruction
        elif augmented_messages and isinstance(augmented_messages[-1].get('content'), list):
            augmented_messages[-1] = dict(augmented_messages[-1])
            augmented_messages[-1]['content'] = list(augmented_messages[-1]['content'])
            augmented_messages[-1]['content'].append({'type': 'text', 'text': json_instruction})
        
        response_text = _local_chat_completion_ra(augmented_messages, max_new_tokens=1024)
        self.track_tokens(len(response_text.split()), response_text)
        args = _extract_json_from_text(response_text)
        return ReasoningActionResponse(**args)

    def define_next_action(self, latest_frame: FrameData) -> ReasoningActionResponse:
        current_grid = latest_frame.frame[-1] if latest_frame.frame else []
        map_image = self.generate_grid_image_with_zone(current_grid, cell_size=8)  # Bug fix #5
        system_prompt = self.build_user_prompt(latest_frame)
        latest_action = self.history[-1] if self.history else None
        user_message_content: List[Dict[str, Any]] = []

        previous_screen = self.screen_history[-1] if self.screen_history else None
        if previous_screen:
            user_message_content.extend([
                {'type': 'text', 'text': 'Previous screen:'},
                {'type': 'image_url', 'image_url': {
                    'url': f'data:image/png;base64,{base64.b64encode(previous_screen).decode()}',
                    'detail': 'high',
                }},
            ])

        # Bug fix #6: inject full aggregated findings, not just latest action
        all_findings = '\n'.join(
            f'- {a.aggregated_findings}' for a in self.history[-5:]
        ) if self.history else 'No findings yet.'

        user_message_text = (
            f'Your previous action was: {json.dumps(latest_action.model_dump() if latest_action else None, indent=2)}\n\n'
            f'## All Accumulated Findings (Bug fix #6):\n{all_findings}\n\n'
            f'What should you do next?'
        )

        current_image_b64 = base64.b64encode(map_image).decode()
        user_message_content.extend([
            {'type': 'text', 'text': user_message_text},
            {'type': 'image_url', 'image_url': {
                'url': f'data:image/png;base64,{current_image_b64}',
                'detail': 'high',
            }},
        ])

        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message_content},
        ]
        result = self.call_llm_with_structured_output(messages)

        self.screen_history.append(map_image)
        if len(self.screen_history) > self.max_screen_history:
            self.screen_history.pop(0)

        return result

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.full_reset:
            self.clear_history()
            return GameAction.RESET

        if not self.history:
            action = GameAction.RESET
            self.history.append(ReasoningActionResponse(
                name='RESET',
                reason='Initial action to start the game.',
                short_description='Start game',
                hypothesis='Game requires RESET to begin.',
                aggregated_findings='No findings yet.',
            ))
            return action

        action_response = self.define_next_action(latest_frame)
        self.history.append(action_response)
        action = GameAction.from_name(action_response.name)

        action.reasoning = {
            'model': self.MODEL,
            'reasoning_effort': self.REASONING_EFFORT,
            'agent_type': 'reasoning_agent_fixed',
            'hypothesis': action_response.hypothesis,
            'aggregated_findings': action_response.aggregated_findings,
            'action_chosen': action.name,
            'game_context': {
                # Bug fix #1: was latest_frame.score (removed in v0.9.3)
                'score': latest_frame.levels_completed,
                'state': latest_frame.state.name,
                'action_counter': self.action_counter,
            },
        }
        return action


Writing /kaggle/working/ARC-AGI-3-Agents/agents/templates/reasoning_agent_fixed.py


## STEP 14 — Write `agents/__init__.py`

Register `HypothesisAgent` and `ReasoningAgentFixed` by name (Section 1.6 + 4.4).

In [14]:
init_code = '''from typing import Type
from dotenv import load_dotenv
load_dotenv()

try:
    from .agent import Agent, Playback
    from .swarm import Swarm
    from .templates.simple_agent import SimpleAgent
    from .templates.hypothesis_agent import HypothesisAgent
    from .templates.reasoning_agent_fixed import ReasoningAgentFixed

    AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
        "simple": SimpleAgent,
        "hypothesisagent": HypothesisAgent,
        "reasoningagent": ReasoningAgentFixed,
    }
    AVAILABLE_AGENTS["hypothesisagent"] = HypothesisAgent
    AVAILABLE_AGENTS["reasoningagent"] = ReasoningAgentFixed

except ModuleNotFoundError as e:
    import warnings
    warnings.warn(f"agents core not fully loaded: {e}. Only utils available.")
    AVAILABLE_AGENTS = {}
'''
with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
    f.write(init_code)
print('agents/__init__.py written.')

agents/__init__.py written.


In [15]:
import sys
for key in list(sys.modules.keys()):
    if "agents" in key:
        del sys.modules[key]

## STEP 15 — Write `.env` configuration

In [16]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    env_code = """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
"""
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(env_code)
    print('.env written for competition mode.')
else:
    print('Local mode: .env not written (use your own .env file).')

Local mode: .env not written (use your own .env file).


In [17]:
import os

# Check competition input files
comp_path = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/'
print('Competition input contents:')
for f in os.listdir(comp_path):
    full = os.path.join(comp_path, f)
    if os.path.isdir(full):
        print(f'  📁 {f}/')
        for sub in os.listdir(full)[:5]:
            print(f'      {sub}')
    else:
        size = os.path.getsize(full)
        print(f'  📄 {f} ({size/1024/1024:.1f} MB)')

# Check if any model is provided
print()
print('Environment variables:')
for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'ARC_API_KEY', 'MODEL_PATH']:
    print(f'  {key}: {os.getenv(key, "NOT SET")}')

Competition input contents:
  📁 environment_files/
      sk48
      tn36
      m0r0
      bp35
      cn04
  📁 ARC-AGI-3-Agents/
      tests
      .pre-commit-config.yaml
      pytest.ini
      LICENSE
      .gitignore
  📁 arc_agi_3_wheels/
      typing_extensions-4.15.0-py3-none-any.whl
      packaging-26.1-py3-none-any.whl
      charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
      fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
      pyparsing-3.3.2-py3-none-any.whl

Environment variables:
  OPENAI_API_KEY: NOT SET
  OPENAI_BASE_URL: NOT SET
  ARC_API_KEY: NOT SET
  MODEL_PATH: NOT SET


## STEP 16 — Smoke test (local mode only)

Validates all imports and module structure without requiring the competition gateway.

In [18]:
import sys
import os

# Only run smoke test in local mode
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== Smoke test: checking imports and core logic (no gateway needed) ===')

    sys.path.insert(0, '/kaggle/working/ARC-AGI-3-Agents')

    import importlib.util

    # Test visual_renderer
    try:
        spec = importlib.util.spec_from_file_location(
            'visual_renderer',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/visual_renderer.py'
        )
        vr = importlib.util.module_from_spec(spec)
        sys.modules['visual_renderer'] = vr
        spec.loader.exec_module(vr)

        grid = [[0] * 64 for _ in range(64)]
        img = vr.grid_to_image(grid)
        print(f'  ✓ grid_to_image: {img.size} (expected (128, 128))')

        b64 = vr.image_to_base64(img)
        print(f'  ✓ image_to_base64: {len(b64)} chars')

        diff = vr.image_diff(img, img)
        print(f'  ✓ image_diff (identical): {diff.size}')

    except Exception as e:
        print(f'  ✗ visual_renderer: {e}')

    # Test frame_diff
    try:
        spec2 = importlib.util.spec_from_file_location(
            'frame_diff',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/frame_diff.py'
        )
        fd = importlib.util.module_from_spec(spec2)
        sys.modules['frame_diff'] = fd
        spec2.loader.exec_module(fd)

        h1 = fd.get_full_hash([[0]*64 for _ in range(64)])
        h2 = fd.get_abstract_hash([[0]*64 for _ in range(64)], set())
        print(f'  ✓ full_hash == abstract_hash (no decorative): {h1 == h2}')

        grid_a = [[0]*64 for _ in range(64)]
        grid_b = [[0]*64 for _ in range(64)]
        grid_b[5][5] = 9
        delta = fd.compute_delta(grid_a, grid_b, 0, 0, 'PLAYING', 'PLAYING', set(), set())
        print(f'  ✓ compute_delta: {delta.cells_changed} cell(s) changed (expected 1)')
        print(f'  ✓ new_values_seen: {delta.new_values_seen} (expected {{9}})')

    except Exception as e:
        print(f'  ✗ frame_diff: {e}')

    # Test rule_book
    try:
        spec3 = importlib.util.spec_from_file_location(
            'rule_book',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/rule_book.py'
        )
        rb_mod = importlib.util.module_from_spec(spec3)
        sys.modules['rule_book'] = rb_mod
        spec3.loader.exec_module(rb_mod)

        rb = rb_mod.RuleBook()
        rb.confirmed_rules = ['ACTION1 moves player up', 'Value 5 is a wall']
        rb.win_condition = 'Reach the yellow tile'
        compact = rb.to_compact_json()
        print(f'  ✓ RuleBook.to_compact_json: {len(compact)} chars')

        rb.update_from_analysis({
            'confirmed_rules': ['Doors open when key collected'],
            'retracted_rules': ['Value 5 is a wall'],
            'new_tentative_rules': ['Yellow value 11 is the goal'],
            'win_condition': 'Reach value 11',
            'current_objective': 'Find and collect key',
        })
        print(f'  ✓ update_from_analysis: confirmed={rb.confirmed_rules}')
        print(f'  ✓ retracted: {rb.retracted_rules}')

    except Exception as e:
        print(f'  ✗ rule_book: {e}')

    # Test pathfinder
    try:
        spec4 = importlib.util.spec_from_file_location(
            'pathfinder',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/pathfinder.py'
        )
        pf = importlib.util.module_from_spec(spec4)
        sys.modules['pathfinder'] = pf
        spec4.loader.exec_module(pf)

        test_grid = [[0]*8 for _ in range(8)]
        test_grid[0][7] = 9
        path = pf.bfs_path(test_grid, (0, 0), 9, [5])
        print(f'  ✓ bfs_path: {path} (expected 7 RIGHT moves)')

    except Exception as e:
        print(f'  ✗ pathfinder: {e}')

    # Test state_graph
    try:
        spec5 = importlib.util.spec_from_file_location(
            'state_graph',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/state_graph.py'
        )
        sg_mod = importlib.util.module_from_spec(spec5)
        sys.modules['state_graph'] = sg_mod
        spec5.loader.exec_module(sg_mod)

        sg = sg_mod.StateGraph()
        sg.add_state('hash_A')
        sg.add_state('hash_B', 'ACTION1', 'hash_A')
        sg.add_state('hash_C', 'ACTION2', 'hash_A')
        untried = sg.get_untried_actions('hash_A', ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
        print(f'  ✓ StateGraph untried from hash_A: {untried} (expected ACTION3+ACTION4)')
        print(f'  ✓ is_dead_end with only 2 available: {sg.is_dead_end("hash_A", ["ACTION1", "ACTION2"])}')

    except Exception as e:
        print(f'  ✗ state_graph: {e}')

    # Test goal_detector
    try:
        spec6 = importlib.util.spec_from_file_location(
            'goal_detector',
            '/kaggle/working/ARC-AGI-3-Agents/agents/utils/goal_detector.py'
        )
        gd = importlib.util.module_from_spec(spec6)
        sys.modules['goal_detector'] = gd
        spec6.loader.exec_module(gd)

        g = [[0]*64 for _ in range(64)]
        g[0][0] = 11
        candidates = gd.detect_goal_candidates([g, g], [], [])
        print(f'  ✓ detect_goal_candidates: {candidates[:2]}')

    except Exception as e:
        print(f'  ✗ goal_detector: {e}')

    print()
    print('=== Smoke test complete. Run STEP 17 on Kaggle competition to play. ===')
else:
    print('Competition mode: skipping smoke test.')

=== Smoke test: checking imports and core logic (no gateway needed) ===
  ✓ grid_to_image: (128, 128) (expected (128, 128))
  ✓ image_to_base64: 392 chars
  ✓ image_diff (identical): (128, 128)
  ✓ full_hash == abstract_hash (no decorative): True
  ✓ compute_delta: 1 cell(s) changed (expected 1)
  ✓ new_values_seen: {0, 9} (expected {9})
  ✓ RuleBook.to_compact_json: 368 chars
  ✓ update_from_analysis: confirmed=['ACTION1 moves player up', 'Doors open when key collected']
  ✓ retracted: ['Value 5 is a wall']
  ✓ bfs_path: ['ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4'] (expected 7 RIGHT moves)
  ✓ StateGraph untried from hash_A: ['ACTION3', 'ACTION4'] (expected ACTION3+ACTION4)
  ✓ is_dead_end with only 2 available: True
  ✓ detect_goal_candidates: [(11, 0.75)]

=== Smoke test complete. Run STEP 17 on Kaggle competition to play. ===


In [19]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/main.py
# ruff: noqa: E402
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env.example")
load_dotenv(dotenv_path=".env", override=True)

import argparse
import json
import logging
import os
import signal
import sys
import threading
from functools import partial
from types import FrameType
from typing import Optional

import requests

from agents import AVAILABLE_AGENTS, Swarm
from agents.tracing import initialize as init_agentops

logger = logging.getLogger()

SCHEME = os.environ.get("SCHEME", "http")
HOST = os.environ.get("HOST", "gateway")
PORT = os.environ.get("PORT", 8001)

if (SCHEME == "http" and str(PORT) == "80") or (
    SCHEME == "https" and str(PORT) == "443"
):
    ROOT_URL = f"{SCHEME}://{HOST}"
else:
    ROOT_URL = f"{SCHEME}://{HOST}:{PORT}"

HEADERS = {
    "X-API-Key": os.getenv("ARC_API_KEY", ""),
    "Accept": "application/json",
}


def run_agent(swarm: Swarm) -> None:
    swarm.main()
    os.kill(os.getpid(), signal.SIGINT)


def cleanup(
    swarm: Swarm,
    signum: Optional[int],
    frame: Optional[FrameType],
) -> None:
    logger.info("Received SIGINT, exiting...")
    card_id = swarm.card_id
    if card_id:
        scorecard = swarm.close_scorecard(card_id)
        if scorecard:
            logger.info("--- EXISTING SCORECARD REPORT ---")
            logger.info(json.dumps(scorecard.model_dump(), indent=2))
            swarm.cleanup(scorecard)
        if card_id:
            scorecard_url = f"{ROOT_URL}/scorecards/{card_id}"
            logger.info(f"View your scorecard online: {scorecard_url}")
    sys.exit(0)


def main() -> None:
    log_level = logging.INFO
    if os.environ.get("DEBUG", "False") == "True":
        log_level = logging.DEBUG

    logger.setLevel(log_level)
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(log_level)
    stdout_handler.setFormatter(formatter)

    file_handler = logging.FileHandler("logs.log", mode="w")
    file_handler.setLevel(log_level)
    file_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stdout_handler)

    parser = argparse.ArgumentParser(description="ARC-AGI-3-Agents")
    parser.add_argument("-a", "--agent", choices=AVAILABLE_AGENTS.keys())
    parser.add_argument("-g", "--game")
    parser.add_argument("-t", "--tags", type=str, default=None)

    args = parser.parse_args()

    if not args.agent:
        logger.error("An Agent must be specified")
        return

    print(f"{ROOT_URL}/api/games")

    full_games = []
    try:
        with requests.Session() as session:
            session.headers.update(HEADERS)
            r = session.get(f"{ROOT_URL}/api/games", timeout=10)
        if r.status_code == 200:
            try:
                full_games = [g["game_id"] for g in r.json()]
            except (ValueError, KeyError) as e:
                logger.error(f"Failed to parse games response: {e}")
                logger.error(f"Response content: {r.text[:200]}")
        else:
            logger.error(f"API request failed with status {r.status_code}: {r.text[:200]}")
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to connect to API server: {e}")

    if not full_games and args.agent and args.agent.endswith(".recording.jsonl"):
        from agents.recorder import Recorder
        game_prefix = Recorder.get_prefix_one(args.agent)
        full_games = [game_prefix]

    games = full_games[:]
    if args.game:
        filters = args.game.split(",")
        games = [g for g in full_games if any(g.startswith(p) for p in filters)]

    logger.info(f"Game list: {games}")

    if not games:
        logger.error("No games available to play. Check API connection.")
        return

    tags: list[str] = []
    if args.tags:
        tags.extend([t.strip() for t in args.tags.split(",")])

    init_agentops(api_key=os.getenv("AGENTOPS_API_KEY"), log_level=log_level)

    swarm = Swarm(args.agent, ROOT_URL, games, tags=tags)
    agent_thread = threading.Thread(target=partial(run_agent, swarm))
    agent_thread.daemon = True
    agent_thread.start()

    signal.signal(signal.SIGINT, partial(cleanup, swarm))

    try:
        while agent_thread.is_alive():
            agent_thread.join(timeout=5)
    except KeyboardInterrupt:
        cleanup(swarm, signal.SIGINT, None)
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        cleanup(swarm, None, None)


if __name__ == "__main__":
    os.environ["TESTING"] = "False"
    main()

Overwriting /kaggle/working/ARC-AGI-3-Agents/main.py


## STEP 17 — Run HypothesisAgent (competition) or dummy submission (local)

In [20]:
## STEP 17 — Run HypothesisAgent (competition) or dummy submission (local)
import os
import sys
import subprocess
import pandas as pd

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== Running HypothesisAgent on competition games ===')
    print('Agent: hypothesisagent')
    print('MAX_ACTIONS: 100 (Section 4.3 bug fix #3)')
    print('Reasoning effort: adaptive per-call (Section 2.9)')
    print()

    # Verify main.py exists
    main_path = '/kaggle/working/ARC-AGI-3-Agents/main.py'
    if not os.path.exists(main_path):
        raise FileNotFoundError(f'main.py not found — make sure the %%writefile cell above ran')

    result = subprocess.run(
        [sys.executable, 'main.py', '-a', 'hypothesisagent'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        env={**os.environ, 'MPLBACKEND': 'agg'},
    )
    print(f'\nmain.py exit code: {result.returncode}')

    # Verify submission was written
    sub_path = '/kaggle/working/submission.parquet'
    if os.path.exists(sub_path):
        df = pd.read_parquet(sub_path)
        print(f'Submission rows: {len(df)}')
        print(df.head())
    else:
        raise FileNotFoundError('submission.parquet was NOT written — check logs above')

else:
    print('Local mode: creating dummy submission parquet.')
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    )
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission.head())
    print()
    print('Ready. Set KAGGLE_IS_COMPETITION_RERUN=1 and deploy to Kaggle.')

Local mode: creating dummy submission parquet.
  row_id game_id  end_of_game  score
0    1_0       1         True      1

Ready. Set KAGGLE_IS_COMPETITION_RERUN=1 and deploy to Kaggle.


---

## Architecture Summary

| Component | File | Section |
|---|---|---|
| Grid rendering (128×128) | `utils/visual_renderer.py` | 1.5, bug fix #5 |
| Frame delta + two-tier hashing | `utils/frame_diff.py` | 2.3, 2.11 |
| Persistent rule accumulator | `utils/rule_book.py` | 2.4 |
| Multi-heuristic goal detection | `utils/goal_detector.py` | 2.6 |
| BFS pathfinder + caching | `utils/pathfinder.py` | 2.7 |
| Stuck-state prevention | `utils/state_graph.py` | 2.5, 2.11 |
| Phase + adaptive effort | `utils/phase_controller.py` | 2.8, 2.9 |
| Prompt templates | `utils/prompt_builder.py` | App A |
| Primary agent (2 calls/action) | `templates/hypothesis_agent.py` | 3, 4.1 |
| Bug-fixed fallback agent | `templates/reasoning_agent_fixed.py` | 4.3 |

**Key metrics targets (Section 6.2):**
- LLM calls per action < 0.35 (most actions use fast path)
- ≥ 60% of LLM calls at `low`/`medium` reasoning effort
- Transition cache hit rate > 15% on looping games
- Goal detection in top-2 within 20 actions on ≥70% of games
- All 55 games complete within 9-hour Kaggle budget